In [1]:
with open('text.txt', 'r', encoding='utf-8') as file:
    faqs = file.read().lower()

In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

In [3]:
tok=Tokenizer()

In [4]:
tok.fit_on_texts([faqs])

In [5]:
total=len(tok.word_index)
print(total)

8931


In [6]:
input_seq=[]
for sen in faqs.split('\n'):
 tok_sen=tok.texts_to_sequences([sen])[0]
 for i in range(1,len(tok_sen)):
  input_seq.append(tok_sen[:i+1])


In [7]:
input_seq

[[145, 4790],
 [145, 4790, 1],
 [145, 4790, 1, 1020],
 [145, 4790, 1, 1020, 4],
 [145, 4790, 1, 1020, 4, 128],
 [145, 4790, 1, 1020, 4, 128, 34],
 [145, 4790, 1, 1020, 4, 128, 34, 45],
 [145, 4790, 1, 1020, 4, 128, 34, 45, 611],
 [145, 4790, 1, 1020, 4, 128, 34, 45, 611, 2235],
 [145, 4790, 1, 1020, 4, 128, 34, 45, 611, 2235, 2236],
 [30, 1021],
 [30, 1021, 15],
 [30, 1021, 15, 23],
 [30, 1021, 15, 23, 1],
 [30, 1021, 15, 23, 1, 275],
 [30, 1021, 15, 23, 1, 275, 4],
 [30, 1021, 15, 23, 1, 275, 4, 394],
 [30, 1021, 15, 23, 1, 275, 4, 394, 2237],
 [30, 1021, 15, 23, 1, 275, 4, 394, 2237, 21],
 [30, 1021, 15, 23, 1, 275, 4, 394, 2237, 21, 51],
 [30, 1021, 15, 23, 1, 275, 4, 394, 2237, 21, 51, 1676],
 [30, 1021, 15, 23, 1, 275, 4, 394, 2237, 21, 51, 1676, 2],
 [30, 1021, 15, 23, 1, 275, 4, 394, 2237, 21, 51, 1676, 2, 18],
 [572, 51],
 [572, 51, 3398],
 [572, 51, 3398, 3399],
 [572, 51, 3398, 3399, 13],
 [572, 51, 3398, 3399, 13, 75],
 [572, 51, 3398, 3399, 13, 75, 817],
 [572, 51, 3398, 33

In [8]:
max_len=max([len(x) for x in input_seq])
print(max_len)

20


In [9]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
padded_input_seq=pad_sequences(input_seq,maxlen=max_len,padding='pre')


In [10]:
padded_input_seq

array([[   0,    0,    0, ...,    0,  145, 4790],
       [   0,    0,    0, ...,  145, 4790,    1],
       [   0,    0,    0, ..., 4790,    1, 1020],
       ...,
       [   0,    0,    0, ...,    3,  360,   83],
       [   0,    0,    0, ...,  360,   83,  358],
       [   0,    0,    0, ...,   83,  358, 1673]], dtype=int32)

In [11]:
x=padded_input_seq[:,:-1]

In [12]:
y=padded_input_seq[:,-1]

In [13]:
x.shape

(101619, 19)

In [14]:
y.shape

(101619,)

In [15]:
from tensorflow.keras.utils import to_categorical

In [16]:
y=to_categorical(y,num_classes=total+1)

In [17]:
y.shape

(101619, 8932)

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense

In [19]:
model = Sequential()
model.add(Embedding(total + 1, 100, input_shape=(max_len-1,)))
model.add(LSTM(150))
model.add(Dense(total + 1, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [20]:
model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

In [21]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 19, 100)        │       893,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 150)            │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8932)           │     1,348,732 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,392,532 (9.13 MB)

 Trainable params: 2,392,532 (9.13 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
model.fit(x,y,epochs=70)

Epoch 1/70
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 33s 9ms/step - accuracy: 0.0742 - loss: 6.2774
Epoch 2/70
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - accuracy: 0.1269 - loss: 5.5037
Epoch 3/70
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - accuracy: 0.1529 - loss: 5.0820
Epoch 4/70
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - accuracy: 0.1730 - loss: 4.7316
Epoch 5/70
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - accuracy: 0.1927 - loss: 4.4152
Epoch 6/70
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - accuracy: 0.2130 - loss: 4.1188
Epoch 7/70
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step - accuracy: 0.2386 - loss: 3.8427
Epoch 8/70
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - accuracy: 0.2718 - loss: 3.5778
Epoch 9/70
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - accuracy: 0.3055 - loss: 3.3338
Epoch 10/70
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - accuracy: 0.3421 - loss: 3.1050
Epoch 11/70
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - accuracy: 0.3775 - loss: 2.8951
Epoch 12/70
3176/31

In [23]:

import numpy as np
text='he said that'
for i in range(5):
  tok_t=tok.texts_to_sequences([text])[0]
  padded_token_t=pad_sequences([tok_t],maxlen=max_len,padding='pre')

  pos = np.argmax(model.predict(padded_token_t), axis=-1)
  for word,index in tok.word_index.items():
    if index==pos:
      text=text+" "+word
      print(text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step
he said that he
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
he said that he became
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
he said that he became his
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
he said that he became his tool
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
he said that he became his tool and


In [24]:
import pickle

model.save("next_word_model.h5")

with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tok, handle, protocol=pickle.HIGHEST_PROTOCOL)
